# **Fairness-Aware Age Estimation: Bias Detection and Mitigation - Part IV**
UB Master in Fundamental Principels of Data Science (2025-2026)

Author: Julio C. S. Jacques Junior

Last modified: Jan, 2026.

---

# **Part IV: Goal**

- Define a metric to **evaluate different kinds of biases** (i.e., age, gender, ethnicity, and emotion biases).
- **Compare the models** trained in the previous notebooks (Part I, II and III).

- **Requirements:** Carefully review the instructions in Notebooks **Part I** through **Part III**.



## Importing required libraries

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import csv
from PIL import Image
from timm import create_model
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy
from tqdm import tqdm

# **Downloading the Appa-Real Dataset**
- Please, check the **detailed instructions in notebook Part I**.

In [ ]:
from zipfile import ZipFile

# downloading the data
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2025/appa-real-dataset_v2.zip

with ZipFile('appa-real-dataset_v2.zip','r') as zip:
   zip.extractall()
   print('Data decompressed successfully')

# removing the .zip file after extraction to clean space
!rm appa-real-dataset_v2.zip

# **Mount Google Drive to save the trained model on the cloud**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
# Note, the default path will be: '/content/gdrive/MyDrive/'
# In my case, the final path will be: '/content/gdrive/MyDrive/temp/' as I
# created a '/temp/' folder in my google drive for this purpose.

# **Defining the Data Loader Class**
- In this example, metadata information is loaded but not used. Future implementations can take benefit of it.
- Note that age labels are divided by 100 (assuming 100 is the max age found in the dataset) so that the age values can be normalized to be in the range of 0 and 1. This way, we can add a sigmoid activation in the last layer of our model.

In [ ]:
class AgeEstimationDataset(Dataset):
    def __init__(self, image_dir, csv_file, transform=None):
        self.image_dir = image_dir
        self.data_info = pd.read_csv(csv_file)
        self.base_transforms = base_transforms
        self.age_normalization_factor = 100; # used to normalize age labels

    def __len__(self):
        return len(self.data_info)

    def __normalization_factor__(self):
        return self.age_normalization_factor

    def __getitem__(self, idx):
        image_id = f"{self.data_info.iloc[idx, 0]:06d}.jpg"  # Format image ID
        image_path = os.path.join(self.image_dir, image_id)
        image = Image.open(image_path).convert("RGB")  # Load image as RGB

        raw_age = float(self.data_info.iloc[idx, 1])
        # normalizing age labes (by 100) to be between 0 and 1 (assuming 100 is the max age)
        age = raw_age / self.age_normalization_factor
        metadata = self.data_info.iloc[idx, 2:].tolist()  # Extract metadata as list

        image = self.base_transforms(image)

        return image, torch.tensor(age, dtype=torch.float32), metadata

# **Defining the base image transformations**

In [ ]:
base_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# **Loading the pretrained ViT baselone model and adapting it to our problem**

- ViT normally outputs class scores for classification tasks; here, we adapt it for regression by setting **num_classes=1**.

> **Note:** This notebook is intended as a starting point. For your deliverables, avoid making only minor modifications. Instead, explore your creativity and try more substantial improvements.

- **It is also recommended to use the same architecture when comparing results with and without data augmentation, in order to ensure a fair comparison.**


In [ ]:
# Vision Transformer Model for Age Prediction (pretrained on ImageNet)
# https://pytorch.org/vision/main/models/vision_transformer.html
# https://huggingface.co/docs/transformers/main/en//model_doc/vit
class AgeEstimationViT(nn.Module):
    def __init__(self):
        super(AgeEstimationViT, self).__init__()
        self.vit = create_model("vit_base_patch16_224", pretrained=True, num_classes=1) # num_classes=1 as we want to regress a single (age value)
        self.activation = nn.Sigmoid()  # Added Sigmoid activation

    def forward(self, x):
        x = self.vit(x)
        return self.activation(x)  # Apply Sigmoid activation

# **Loading the Train and Validation sets**

In [ ]:
# Create dataset and dataloader (train set):
dataset_train = AgeEstimationDataset("train_data", "labels_metadata_train.csv", transform=base_transforms)
dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True, num_workers=2)
print(f"Total number of train samples: {len(dataloader_train.dataset)}")

# Create dataset and dataloader (validation set):
dataset_valid = AgeEstimationDataset("valid_data", "labels_metadata_valid.csv", transform=base_transforms)
dataloader_valid = DataLoader(dataset_valid, batch_size=32, shuffle=True, num_workers=2)
print(f"Total number of valid samples: {len(dataloader_valid.dataset)}")

# **Defining an auxiliary function to evaluate the model**
- Check Part I for the details.

In [ ]:
# Function to make predictions on test set and compute MSE
def predict_and_evaluate(model_path, test_dataset, output_zip=None, batch_size=32, output_csv="predictions.csv"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AgeEstimationViT().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    predictions = []
    actual_ages = []

    with torch.no_grad():
        for images, labels, metadata in tqdm(test_loader, desc="Predicting"):
            images = images.to(device)
            outputs = model(images).squeeze().cpu().numpy()
            labels = labels.cpu().numpy()

            predictions.extend(outputs * test_dataset.__normalization_factor__())
            actual_ages.extend(labels * test_dataset.__normalization_factor__())

    mae = np.mean(np.abs(np.array(predictions) - np.array(actual_ages)))
    if output_zip is not None:
      print(f"\n=======\nMean Absolute Error on Test Set: {mae:.4f}")
    else:
      print(f"\n=======\nMean Absolute Error on Validation Set: {mae:.4f}")


    # Only create ZIP if output_zip is provided
    if output_zip is not None:
        # Save predictions to CSV without headers
        with open(output_csv, mode='w', newline='') as file:
            writer = csv.writer(file)
            for pred in predictions:
                writer.writerow([pred])

        with ZipFile(output_zip, 'w') as zipf:
            zipf.write(output_csv, os.path.basename(output_csv))
        print(f"Predictions saved to {output_csv} and compressed as {output_zip}")

    return predictions, mae

# **Loading the Saved Models and Making Predictions on the Validation Set**
- First, we donload the models from our Google Drive
 >- **Model 1:** without data augmentation, without custom loss
 >- **Model 2:** with data augmentation, without custom loss
 >- **Model 3:** with custom loss, without data augmentation

In [ ]:
model_1_filename = "/content/gdrive/MyDrive/temp/best_age_estimation_model.pth"
model_2_filename = "/content/gdrive/MyDrive/temp/best_age_estimation_model_aug.pth"
model_3_filename = "/content/gdrive/MyDrive/temp/best_age_estimation_model_custom_loss.pth"

# Run prediction and compute MAE
predictions_1, mae_1 = predict_and_evaluate(model_1_filename, dataset_valid,output_zip=None, batch_size=32,output_csv=None)
predictions_2, mae_2 = predict_and_evaluate(model_2_filename, dataset_valid,output_zip=None, batch_size=32,output_csv=None)
predictions_3, mae_3 = predict_and_evaluate(model_3_filename, dataset_valid,output_zip=None, batch_size=32,output_csv=None)

---
# **Defining our Bias Metric**
- Accuracy is not enough! We also need to evaluate how biased our model is!
- Next, we define different different functions, used to compute a bias score given different attributes.
  - **Age bias** (given 4 sub-groups, or age ranges)
  - **Gender bias** (given 2 sub-groups)
  - **Ethnicity bias** (given 3 sub-groups)
  - **Facial Expression bias** (given 4 sub-groups)
- In a nutshell, given a particular attribute $A$, we compute the Mean Absolute Error $E_n$ for its $N$ sub-groups. We illustrate this process for the case of **Age** next:
- Consider we have 4 sub-groups base on different age ranges (i.e., "0-19", "20-39", "40-59", and "60-100"). We will have one error value per sub-group: $E_1$, $E_2$, $E_3$ and $E_4$.
- Then, we compute the absolute difference among them all. Consider $D$ is a squared matrix where each element $(i,j)$ is the absolute difference between $E_i$ and $E_j$. Then we can retrieve:
  - $D_{2,1} = |E_1-E_2|$
  - $D_{3,1} = |E_1-E_3|$
  - $D_{4,1} = |E_1-E_4|$
  - $D_{3,2} = |E_2-E_3|$
  - $D_{4,2} = |E_2-E_4|$
  - $D_{4,3} = |E_3-E_4|$

- The Final Bias score $B_A$ for attribute $A$ is obtained by the Average of the computed differences. That is:
  - $B_A = \frac{(D_{2,1} + D_{3,1} + D_{4,1} + D_{3,2} + D_{4,2} + D_{4,3})}{6}$
  - In other words:

   $B_A = \frac{1}{(N^2-N)/2}\sum_{i=1}^{N} \sum_{j=1}^{N} |E_i - E_j|, \forall i,j \in \mathbb{N}^*, \text{if } i < j$

- To minimize your bias score, given a particular attribute, you will need to minimize the absolute difference among the different sub-groups being evaluated. That is, part of your goal will be to make the $N$ sub-groups of each attribute $A$ to have similar errors $E_n$.
- The big challenge here is to minimize **ALL** bias scores (i.e., age, gender, ethnicity and face expression) together.
- Next, we briefly detail the different attributes (and their sub-groups) where the bias scores are evaluated.


## Age Bias ($B_a$)

- Evaluates how accurate the model is with respect to different age ranges.
  - sub-group 1: age < 20
  - sub-group 2: 20 <= age < 40
  - sub-group 3: 40 <= age < 60
  - sub-group 4: 60 <= age

## Gender Bias ($B_g$)
- Evaluates how accurate the model is with respect to different gender.
  - sub-group 1: male
  - sub-group 2: female

## Ethnicity Bias ($B_e$)
- Evaluates how accurate the model is with respect to different ethnicity categories.
  - sub-group 1: asian
  - sub-group 2: afroamerican
  - sub-group 3: caucasian

## Face expression bias ($B_f$)
- Evaluates how accurate the model is with respect to different face expression categories.
  - sub-group 1: neutral
  - sub-group 2: slightlyhappy
  - sub-group 3: happy
  - sub-group 4: other

---
# **Computing Age, Gender, Ethnicity, and Emotion Biases on the Validation Set**

- Next, we compute the different bias scores **using the models trained in the previous notebooks (Part I to Part III)**.
- We re-scale the predictions and labels back to the range of **ages** using the normalization factor defined earlier, in order to make the analysis easier.
- First, we need to **download our "bias library"**, which contains the functions used to evaluate the different bias scores:


In [ ]:
# downloading our "bias library", which contains the functions used to evaluate
# the different bias scores
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/bias_functions.py

# importing the functions used to evaluate the different biases
from bias_functions import age_bias, gender_bias, ethnicity_bias, face_expression_bias

# **Extracting Age Labels and Metadata from the CSV File (Validation Set)**

- This is used to compute the bias metrics.


In [ ]:
import pandas as pd
import numpy as np

# Load the CSV
df = pd.read_csv("labels_metadata_valid.csv")

# Extract age column as floats
ages_validation = df["age"].to_numpy()

# Extract metadata columns as strings
metadata_validation = df[["gender", "ethnicity", "emotion"]].to_numpy(dtype=str)

print("ages shape:", ages_validation.shape)
print("metadata shape:", metadata_validation.shape)

# Visualizing the first few entries
print(ages_validation[:5])
print(metadata_validation[:5])


# **Computing the Age Bias for All Models**

- Note that for each attribute, the number of samples and the MAE per subgroup are also reported, which can be helpful for analysis and discussion.


In [ ]:
# computing the age bias
age_bias(predictions_1,ages_validation)
age_bias(predictions_2,ages_validation)
age_bias(predictions_3,ages_validation)

# **Computing the Gender Bias for All Models**

In [ ]:
# computing the gender bias
gender_bias(predictions_1,ages_validation,metadata_validation)
gender_bias(predictions_2,ages_validation,metadata_validation)
gender_bias(predictions_3,ages_validation,metadata_validation)

# **Computing the Ethnicity Bias for All Models**

In [ ]:
# computing the ethnicity bias
ethnicity_bias(predictions_1,ages_validation,metadata_validation)
ethnicity_bias(predictions_2,ages_validation,metadata_validation)
ethnicity_bias(predictions_3,ages_validation,metadata_validation)

# **Computing the Emotion Bias for All Models**

In [ ]:
# computing the face bias
face_expression_bias(predictions_1,ages_validation,metadata_validation)
face_expression_bias(predictions_2,ages_validation,metadata_validation)
face_expression_bias(predictions_3,ages_validation,metadata_validation)

# **Final Analysis and Discussion of the Results**

- Now, you just need to **generate a clear and well-organized table to summarize the results** before analyzing and discussing them.

- Note that the results (bias metrics, together with the overall MAE) can be used to **refine your proposed strategy** (for example, by changing the data augmentation method to further improve fairness).

- Also note that, **due to variations in initialization**, each training execution can produce slightly different results. To strengthen the robustness of your evaluation, **you may train each model multiple times and report summary statistics**; however, this remains **optional**.

- **Do not focus only on global MAE. Keep in mind that, in addition to being accurate (low MAE), the model should also produce fair predictions (low bias scores).**

- Finally, **after defining the best model for each strategy using the validation set**, you can discuss their strengths and limitations in your report, **using the final results obtained on the Test set**.
